# Inferential Analysis (Statistical Analysis)

**Purpose:** Validate assumptions, check relationships between variables, and make inferences.

**Tasks:**
1. Hypothesis testing (e.g., T-test, ANOVA, Chi-squared test) for statistical significance between groups or variables.
2. Correlation testing (e.g., Pearson, Spearman correlation) to measure the strength and direction of relationships between numerical variables.
3. P-value and confidence intervals for estimating uncertainty and statistical significance.
4. Chi-squared test for independence (in case of categorical variables).
5. Skewness/Kurtosis to check for normality of data.

In [11]:
# importing libraries
import json
import numpy as np
import pandas as pd
import itertools
from scipy import stats
import statsmodels.api as sm
import statsmodels.stats.api as sms
from scipy.stats import chi2_contingency

In [2]:
# Replace with your dataset
df = pd.read_csv(r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\data\raw\fraud test.csv")
data = df[df.columns[1:]].copy()
data.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,21/06/2020 12:14,2.291160e+15,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,...,33.9659,-80.9355,333497,Mechanical engineer,19/03/1968,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,21/06/2020 12:14,3.573030e+15,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,...,40.3207,-110.4360,302,"Sales professional, IT",17/01/1990,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,21/06/2020 12:14,3.598220e+15,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,Bellmore,...,40.6729,-73.5365,34496,"Librarian, public",21/10/1970,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,21/06/2020 12:15,3.591920e+15,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,Titusville,...,28.5697,-80.8191,54767,Set designer,25/07/1987,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,21/06/2020 12:15,3.526830e+15,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,Falmouth,...,44.2529,-85.0170,1126,Furniture designer,06/07/1955,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0


In [3]:
file_path = r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\notebooks\metadata\Credit_cols_classified.json"
with open(file_path, 'r') as file:
    col_class = json.load(file)
print(col_class)

{'target_col': 'is_fraud', 'uniq_cols': ['cc_num', 'trans_num'], 'num_cols': ['amt', 'city_pop'], 'cat_cols': ['merchant', 'category', 'first', 'last', 'gender', 'street', 'city', 'state', 'job'], 'loc_cols': ['long', 'lat', 'merch_lat', 'merch_long', 'zip'], 'time_cols': ['trans_date_trans_time', 'dob', 'unix_time']}


In [4]:
target_col = col_class["target_col"]
num_cols = col_class["num_cols"]
cat_cols = col_class["cat_cols"]
loc_cols = col_class["loc_cols"]
time_cols = col_class["time_cols"]

print("Target:", target_col)
print("Numerical:", num_cols)
print("Categorical:", cat_cols)

Target: is_fraud
Numerical: ['amt', 'city_pop']
Categorical: ['merchant', 'category', 'first', 'last', 'gender', 'street', 'city', 'state', 'job']


**NUMERICAL vs TARGET (T-test & Mann–Whitney)**

*H₀ (Null Hypothesis):*
The mean (or distribution) of the feature is the same for fraud and non-fraud transactions.

*H₁ (Alternative Hypothesis):*
The mean (or distribution) is different between fraud and non-fraud.

In [5]:
results_num_target = []
for col in num_cols:
    fraud = df[df[target_col] == 1][col].dropna()
    non_fraud = df[df[target_col] == 0][col].dropna()
    # T-test
    t_stat, t_p = stats.ttest_ind(fraud, non_fraud, equal_var=False)
    # Mann-Whitney U
    u_stat, u_p = stats.mannwhitneyu(fraud, non_fraud, alternative='two-sided')
    results_num_target.append([col, t_p, u_p])

pd.DataFrame(results_num_target, columns=["Feature", "T-test_p", "MannWhitney_p"])

,Feature,T-test_p,MannWhitney_p
0,amt,0.000000e+00,0.000000
1,city_pop,5.974603e-07,0.004913


**Interpretation:**
- All p-values < 0.05
- Reject H₀ for both features

**Conclusion:**

- amt (transaction amount) is significantly different between fraud and non-fraud
- city_pop also shows a statistically significant difference

**Business Insight:**

Fraud transactions tend to differ in amount and occur in areas with different population sizes

**CATEGORICAL vs TARGET (Chi-Square / Fisher)**

**Hypothesis:**

For each categorical feature:

H₀: Feature is independent of fraud (is_fraud)\
H₁: Feature is associated with fraud

In [6]:
results_cat_target = []

for col in cat_cols:
    contingency = pd.crosstab(df[col], df[target_col])
    chi2, chi_p, _, _ = stats.chi2_contingency(contingency)
    # Fisher only for 2x2
    if contingency.shape == (2,2):
        _, fisher_p = stats.fisher_exact(contingency)
    else:
        fisher_p = np.nan
    results_cat_target.append([col, chi_p, fisher_p])

pd.DataFrame(results_cat_target, columns=["Feature", "Chi2_p", "Fisher_p"])

,Feature,Chi2_p,Fisher_p
0,merchant,1.802227e-205,NaN
1,category,0.000000e+00,NaN
2,first,0.000000e+00,NaN
3,last,0.000000e+00,NaN
4,gender,5.922845e-01,0.586862
5,street,0.000000e+00,NaN
6,city,0.000000e+00,NaN
7,state,6.739749e-66,NaN
8,job,0.000000e+00,NaN


**Results Summary:**

**Significant Features (p < 0.05)**
- merchant
- category
- first
- last
- street
- city
- state
- job

**Not Significant**

gender (p ≈ 0.59)

**Interpretation**

**Reject H₀ (Strong Association with Fraud)**

**For:**
- merchant, category → VERY important fraud indicators
- location fields (city, state, street) → fraud depends on geography
- job, name → may indicate behavioral or demographic patterns

**Fail to Reject H₀**
- gender → No statistical relationship with fraud

**NUMERICAL vs NUMERICAL (Correlation)**

**Hypothesis**

For correlation between amt and city_pop:

- H₀: No relationship (correlation = 0)
- H₁: There is a relationship

In [7]:
results_corr = []

for col1, col2 in itertools.combinations(num_cols, 2):
    df_sub = df[[col1, col2]].dropna()
    pearson = stats.pearsonr(df_sub[col1], df_sub[col2])[1]
    spearman = stats.spearmanr(df_sub[col1], df_sub[col2])[1]
    kendall = stats.kendalltau(df_sub[col1], df_sub[col2])[1]
    results_corr.append([col1, col2, pearson, spearman, kendall])

pd.DataFrame(results_corr, columns=["Var1", "Var2", "Pearson_p", "Spearman_p", "Kendall_p"])

,Var1,Var2,Pearson_p,Spearman_p,Kendall_p
0,amt,city_pop,0.039669,8.169248e-74,7.342026e-74


**Interpretation**

- All p-values < 0.05
- Reject H₀ → relationship exists

**BUT Important nuance:**

- Pearson p = 0.039 → weak linear relationship
- Spearman/Kendall → extremely significant → strong monotonic relationship

**Conclusion**

- amt and city_pop are statistically related
- But likely:
    - Not strongly linear
    - Possibly skewed or non-linear relationship

### Chi-Square Test of Independence

**Hypothesis**

**H0:** category and is_fraud are independent\
**H1:** They are associated

In [8]:
cont_table = pd.crosstab(data["category"], data["is_fraud"])

chi2, p, dof, expected = chi2_contingency(cont_table)

print("Chi2:", chi2)
print("p-value:", p)

Chi2: 1852.2418680553296
p-value: 0.0


**Interpretation**

p < 0.05 → category significantly affects fraud

**Note:** Chi-square tells existence of relationship, not strength

In [9]:
from statsmodels.stats.proportion import proportions_ztest

cat1 = data[data["category"] == "grocery_pos"]["is_fraud"]
cat2 = data[data["category"] == "shopping_net"]["is_fraud"]

count = [cat1.sum(), cat2.sum()]
nobs = [len(cat1), len(cat2)]

stat, p = proportions_ztest(count, nobs)
print(p)

1.6080435133331956e-05


**FINAL SUMMARY (What This Means)**

**Strong Fraud Indicators**

- amt
- category, merchant
- location (city, state)

**Weak / No Signal**
- gender 
 
**Feature Relationships**
- amt ↔ city_pop → related but not strongly linear